<a href="https://colab.research.google.com/github/ThLanBot42/DHU_AI_THL_CV_COMP_COMPETITION/blob/main/Firegazing_V4_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 【Cell 0】创建存储结构 —— 自动挂载Drive（防御性）
import os

# ========== 先检查/挂载 Drive ==========
if not os.path.exists('/content/drive/MyDrive'):
    print("⚠️ Drive 未挂载，自动挂载中...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Drive 挂载完成")
else:
    print("✅ Drive 已挂载")

# ========== 项目根目录 ==========
ROOT = '/content/drive/MyDrive/Wheat'

# ========== 所有目录配置 ==========
ALL_DIRS = [
    # 0_raw: 官方原始数据（你手动上传）
    f'{ROOT}/0_raw/images/A_train',
    f'{ROOT}/0_raw/images/A_test',
    f'{ROOT}/0_raw/images/B_test',
    f'{ROOT}/0_raw/labels/A_train',

    # 1_processed: 代码处理后数据
    f'{ROOT}/1_processed/train/images',
    f'{ROOT}/1_processed/train/labels',
    f'{ROOT}/1_processed/val/images',
    f'{ROOT}/1_processed/val/labels',
    f'{ROOT}/1_processed/A_test/images',
    # 5-Fold
    f'{ROOT}/1_processed/folds/fold0/train/images',
    f'{ROOT}/1_processed/folds/fold0/train/labels',
    f'{ROOT}/1_processed/folds/fold0/val/images',
    f'{ROOT}/1_processed/folds/fold0/val/labels',
    f'{ROOT}/1_processed/folds/fold1/train/images',
    f'{ROOT}/1_processed/folds/fold1/train/labels',
    f'{ROOT}/1_processed/folds/fold1/val/images',
    f'{ROOT}/1_processed/folds/fold1/val/labels',
    f'{ROOT}/1_processed/folds/fold2/train/images',
    f'{ROOT}/1_processed/folds/fold2/train/labels',
    f'{ROOT}/1_processed/folds/fold2/val/images',
    f'{ROOT}/1_processed/folds/fold2/val/labels',
    f'{ROOT}/1_processed/folds/fold3/train/images',
    f'{ROOT}/1_processed/folds/fold3/train/labels',
    f'{ROOT}/1_processed/folds/fold3/val/images',
    f'{ROOT}/1_processed/folds/fold3/val/labels',
    f'{ROOT}/1_processed/folds/fold4/train/images',
    f'{ROOT}/1_processed/folds/fold4/train/labels',
    f'{ROOT}/1_processed/folds/fold4/val/images',
    f'{ROOT}/1_processed/folds/fold4/val/labels',

    # 4_output: 模型输出
    f'{ROOT}/4_output',
    f'{ROOT}/4_output/baseline',
    f'{ROOT}/4_output/A_baseline',
    f'{ROOT}/4_output/A_v8x_single',
    f'{ROOT}/4_output/A_fold0',
    f'{ROOT}/4_output/A_fold1',
    f'{ROOT}/4_output/A_fold2',
    f'{ROOT}/4_output/A_fold3',
    f'{ROOT}/4_output/A_fold4',
    f'{ROOT}/4_output/visualizations',

    # 3_manual: 人工标注备用
    f'{ROOT}/3_manual/seed_to_label',
    f'{ROOT}/3_manual/corrected/images',
    f'{ROOT}/3_manual/corrected/labels',
]

# ========== 创建 ==========
created = 0
existed = 0
for d in ALL_DIRS:
    if os.path.exists(d):
        existed += 1
    else:
        os.makedirs(d, exist_ok=True)
        created += 1

print("=" * 50)
print("【Cell 0】存储结构创建完成")
print("=" * 50)
print(f"新建: {created} 个")
print(f"已有: {existed} 个")
print(f"总计: {len(ALL_DIRS)} 个路径")
print("\n【下一步】把官方数据上传到 0_raw/：")
print(f"  • 训练图片 → {ROOT}/0_raw/images/A_train/")
print(f"  • 训练标签 → {ROOT}/0_raw/labels/A_train/")
print(f"  • A榜测试  → {ROOT}/0_raw/images/A_test/")
print(f"  • B榜测试  → {ROOT}/0_raw/images/B_test/")
print(f"  • data.yaml → {ROOT}/0_raw/")

⚠️ Drive 未挂载，自动挂载中...


MessageError: Error: credential propagation was unsuccessful

In [1]:
# 【Cell 1】挂载 Google Drive —— 断联后必须先运行这个
from google.colab import drive
drive.mount('/content/drive')

# 验证挂载成功
import os
assert os.path.exists('/content/drive/MyDrive'), "❌ 挂载失败，请重新运行本 Cell"
print("✅ Google Drive 挂载成功")

Mounted at /content/drive
✅ Google Drive 挂载成功


In [2]:
# 【Cell 2】安装依赖 —— 断联后必须先运行这个
!pip install -q ultralytics
!pip install -q torchvision  # 用于 Ensemble NMS

import ultralytics
print(f"✅ ultralytics 版本: {ultralytics.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ ultralytics 版本: 8.4.47


In [ ]:
# 【Cell 3】全局配置 —— 可重复运行，无副作用
import os, shutil, random, json, time
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

# ========== 路径配置 ==========
ROOT = '/content/drive/MyDrive/Wheat'
OFFICIAL = f'{ROOT}/0_raw'

# 自动恢复：如果 Drive 没挂载，自动挂（防御性）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print("⚠️ 自动挂载了 Drive，下次建议先运行 Cell 1")

# 自动恢复：如果依赖丢了，自动装（防御性）
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics
    from ultralytics import YOLO

# ========== 创建目录（幂等） ==========
for d in [
    f'{ROOT}/1_processed/train/images', f'{ROOT}/1_processed/train/labels',
    f'{ROOT}/1_processed/val/images',   f'{ROOT}/1_processed/val/labels',
    f'{ROOT}/1_processed/A_test/images',
    f'{ROOT}/4_output'
]:
    os.makedirs(d, exist_ok=True)

# ========== 检查原始数据 ==========
train_png = sorted(Path(f'{OFFICIAL}/images/A_train').glob('*.png'))
train_txt = sorted(Path(f'{OFFICIAL}/labels/A_train').glob('*.txt'))
a_test_png = sorted(Path(f'{OFFICIAL}/images/A_test').glob('*.png'))

print(f"【原始数据检查】")
print(f"  A_train 图片: {len(train_png)} 张")
print(f"  A_train 标签: {len(train_txt)} 个")
print(f"  A_test 图片: {len(a_test_png)} 张")

assert len(train_png) == len(train_txt), "❌ 图片和标签数量不匹配！"
assert len(train_png) > 0, "❌ A_train 为空，请检查 0_raw/ 路径"
print(f"✅ 原始数据就绪，可以进入任意阶段")

【原始数据检查】
  A_train 图片: 4116 张
  A_train 标签: 4116 个
  A_test 图片: 0 张
✅ 原始数据就绪，可以进入任意阶段


In [ ]:
# 【Cell 4】阶段1：数据划分（80/20）—— 断联后可恢复，已划分则跳过
# 自动恢复头
import os, shutil, random
from pathlib import Path
ROOT = '/content/drive/MyDrive/Wheat'
OFFICIAL = f'{ROOT}/0_raw'

# 防御性挂载与依赖
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics

# ========== 恢复检查：是否已划分？ ==========
train_imgs_exist = list(Path(f'{ROOT}/1_processed/train/images').glob('*.png'))
val_imgs_exist = list(Path(f'{ROOT}/1_processed/val/images').glob('*.png'))

if len(train_imgs_exist) > 3000 and len(val_imgs_exist) > 800:
    print(f"⚡️ 检测到已划分数据（train:{len(train_imgs_exist)}, val:{len(val_imgs_exist)}）")
    print("✅ 阶段1已完成，跳过。如需强制重分，请手动删除 1_processed/ 下文件后重跑")
else:
    print("🚀 开始数据划分...")

    a_train = sorted(Path(f'{OFFICIAL}/images/A_train').glob('*.png'))
    random.seed(42)
    idx = list(range(len(a_train)))
    random.shuffle(idx)
    n_val = int(len(a_train) * 0.2)

    val_idx, train_idx = set(idx[:n_val]), set(idx[n_val:])

    # 清理旧数据（如果存在残缺的）
    for split in ['train', 'val']:
        for f in Path(f'{ROOT}/1_processed/{split}/images').glob('*'):
            f.unlink()
        for f in Path(f'{ROOT}/1_processed/{split}/labels').glob('*'):
            f.unlink()

    # 复制
    for split_name, split_idx in [('train', train_idx), ('val', val_idx)]:
        for i in split_idx:
            img = a_train[i]
            lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
            shutil.copy(img, f'{ROOT}/1_processed/{split_name}/images/')
            if os.path.exists(lbl):
                shutil.copy(lbl, f'{ROOT}/1_processed/{split_name}/labels/')

    # 复制 A_test（仅图片）
    for img in Path(f'{OFFICIAL}/images/A_test').glob('*.png'):
        shutil.copy(img, f'{ROOT}/1_processed/A_test/images/')

    print(f"✅ 划分完成: train {len(train_idx)}, val {len(val_idx)}")

# ========== 严格验证 ==========
for split in ['train', 'val']:
    imgs = {p.stem for p in Path(f'{ROOT}/1_processed/{split}/images').glob('*.png')}
    lbls = {p.stem for p in Path(f'{ROOT}/1_processed/{split}/labels').glob('*.txt')}
    assert imgs == lbls, f"❌ {split} 不匹配！缺标签: {imgs-lbls}"
    print(f"✅ {split}: {len(imgs)} 对，验证通过")

# 生成 data_fixed.yaml
with open(f'{ROOT}/data_fixed.yaml', 'w') as f:
    f.write(f"path: {ROOT}/1_processed\ntrain: train/images\nval: val/images\nnames:\n  0: wheat_head\nnc: 1")
print(f"✅ data_fixed.yaml 已就绪")

🚀 开始数据划分...


In [1]:
# 【Cell 4 续传版】中断后补救 —— 智能检测，只补缺失
import os, shutil, random
from pathlib import Path

ROOT = '/content/drive/MyDrive/Wheat'
OFFICIAL = f'{ROOT}/0_raw'

# 防御性挂载
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

# ========== 检查当前进度 ==========
train_imgs = list(Path(f'{ROOT}/1_processed/train/images').glob('*.png'))
val_imgs = list(Path(f'{ROOT}/1_processed/val/images').glob('*.png'))
train_lbls = list(Path(f'{ROOT}/1_processed/train/labels').glob('*.txt'))
val_lbls = list(Path(f'{ROOT}/1_processed/val/labels').glob('*.txt'))

print(f"当前进度: train {len(train_imgs)}张/{len(train_lbls)}标签, val {len(val_imgs)}张/{len(val_lbls)}标签")

# ========== 智能补救 ==========
if len(train_imgs) >= 3290 and len(val_imgs) >= 820 and len(train_lbls) >= 3290 and len(val_lbls) >= 820:
    print("⚡️ 数据已基本完整，只补缺失...")
    # 只补缺失，不重新全量复制
    a_train = sorted(Path(f'{OFFICIAL}/images/A_train').glob('*.png'))
    random.seed(42)
    idx = list(range(len(a_train)))
    random.shuffle(idx)
    n_val = int(len(a_train) * 0.2)
    val_idx_set = set(idx[:n_val])
    train_idx_set = set(idx[n_val:])

    # 补train缺失
    for i in train_idx_set:
        img = a_train[i]
        dst_img = f'{ROOT}/1_processed/train/images/{img.name}'
        dst_lbl = f'{ROOT}/1_processed/train/labels/{img.stem}.txt'
        src_lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
        if not os.path.exists(dst_img):
            shutil.copy(img, dst_img)
        if not os.path.exists(dst_lbl) and os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

    # 补val缺失
    for i in val_idx_set:
        img = a_train[i]
        dst_img = f'{ROOT}/1_processed/val/images/{img.name}'
        dst_lbl = f'{ROOT}/1_processed/val/labels/{img.stem}.txt'
        src_lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
        if not os.path.exists(dst_img):
            shutil.copy(img, dst_img)
        if not os.path.exists(dst_lbl) and os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

    print("✅ 缺失补完")

else:
    print("🔄 数据不完整，清理后重新复制...")
    # 清理重来
    for split in ['train', 'val']:
        for f in Path(f'{ROOT}/1_processed/{split}/images').glob('*'):
            f.unlink()
        for f in Path(f'{ROOT}/1_processed/{split}/labels').glob('*'):
            f.unlink()

    # 重新执行完整复制（原Cell 4逻辑）
    a_train = sorted(Path(f'{OFFICIAL}/images/A_train').glob('*.png'))
    random.seed(42)
    idx = list(range(len(a_train)))
    random.shuffle(idx)
    n_val = int(len(a_train) * 0.2)
    val_idx, train_idx = set(idx[:n_val]), set(idx[n_val:])

    for split_name, split_idx in [('train', train_idx), ('val', val_idx)]:
        for i in split_idx:
            img = a_train[i]
            lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
            shutil.copy(img, f'{ROOT}/1_processed/{split_name}/images/')
            if os.path.exists(lbl):
                shutil.copy(lbl, f'{ROOT}/1_processed/{split_name}/labels/')

    print(f"✅ 重新划分完成: train {len(train_idx)}, val {len(val_idx)}")

# ========== 复制A_test（幂等） ==========
for img in Path(f'{OFFICIAL}/images/A_test').glob('*.png'):
    dst = f'{ROOT}/1_processed/A_test/images/{img.name}'
    if not os.path.exists(dst):
        shutil.copy(img, dst)

# ========== 验证 ==========
for split in ['train', 'val']:
    imgs = {p.stem for p in Path(f'{ROOT}/1_processed/{split}/images').glob('*.png')}
    lbls = {p.stem for p in Path(f'{ROOT}/1_processed/{split}/labels').glob('*.txt')}
    assert imgs == lbls, f"❌ {split} 不匹配！缺: {imgs-lbls}"
    print(f"✅ {split}: {len(imgs)} 对")

# 生成yaml
with open(f'{ROOT}/data_fixed.yaml', 'w') as f:
    f.write(f"path: {ROOT}/1_processed\ntrain: train/images\nval: val/images\nnames:\n  0: wheat_head\nnc: 1")
print("✅ data_fixed.yaml 就绪")

Mounted at /content/drive
当前进度: train 2943张/2942标签, val 0张/0标签
🔄 数据不完整，清理后重新复制...
✅ 重新划分完成: train 3293, val 823
✅ train: 3293 对
✅ val: 823 对
✅ data_fixed.yaml 就绪


In [4]:
# 【Cell 5 修复版】基线训练（YOLOv8m）— 已包含所有必要导入
import os
import shutil
from pathlib import Path
from ultralytics import YOLO

ROOT = '/content/drive/MyDrive/Wheat'
BASELINE_DIR = f'{ROOT}/4_output/A_baseline'
BASELINE_WEIGHT = f'{BASELINE_DIR}/weights/best.pt'

# 防御性挂载
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 防御性依赖
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics
    from ultralytics import YOLO

# ========== 恢复检查 ==========
if os.path.exists(BASELINE_WEIGHT) and os.path.getsize(BASELINE_WEIGHT) > 1000000:
    print(f"⚡️ 基线模型已存在，跳过训练")
else:
    print("🚀 开始基线训练...")
    if os.path.exists(BASELINE_DIR):
        shutil.rmtree(BASELINE_DIR)

    model = YOLO('yolov8m.pt')

    model.train(
        data=f'{ROOT}/data_fixed.yaml',
        epochs=100,
        imgsz=640,
        batch=32,
        lr0=0.01,
        lrf=0.01,
        patience=30,
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.2,
        degrees=15.0,
        scale=0.5,
        flipud=0.5,
        fliplr=0.5,
        project=f'{ROOT}/4_output',
        name='A_baseline',
        exist_ok=True
    )

    print("✅ 基线训练完成")

# ========== 验证基线效果 ==========
print("\n📊 验证基线模型...")
model = YOLO(BASELINE_WEIGHT)
metrics = model.val(data=f'{ROOT}/data_fixed.yaml')
print(f"Val mAP50:    {metrics.box.map50:.4f}")
print(f"Val mAP50-95: {metrics.box.map:.4f}")

if metrics.box.map50 < 0.5:
    print("⚠️ mAP50 < 0.5，管道可能有问题，建议检查标签格式")
elif metrics.box.map > 0.55:
    print("🎯 基线很强，可以进入阶段3（YOLOv8x）")
else:
    print("✅ 基线正常，建议继续阶段3压榨极限")

🚀 开始基线训练...
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Wheat/data_fixed.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=A_baseline, nbs=64, nms=False, opset=None, optimize=False, opt

In [4]:
# 【Cell 6】阶段3：单模型极限（YOLOv8x）—— 断联后可恢复
import os, shutil
from pathlib import Path
from ultralytics import YOLO

ROOT = '/content/drive/MyDrive/Wheat'
V8X_DIR = f'{ROOT}/4_output/A_v8x_single'
V8X_WEIGHT = f'{V8X_DIR}/weights/best.pt'

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

# ========== 恢复检查 ==========
if os.path.exists(V8X_WEIGHT) and os.path.getsize(V8X_WEIGHT) > 1000000:
    print(f"⚡️ 检测到 v8x 模型已存在")
    print("✅ 阶段3已完成，跳过")
else:
    print("🚀 开始 YOLOv8x 极限训练...")
    if os.path.exists(V8X_DIR):
        shutil.rmtree(V8X_DIR)

    model = YOLO('yolov8x.pt')
    model.train(
        data=f'{ROOT}/data_fixed.yaml',
        epochs=150,
        imgsz=640,
        batch=16,           # v8x更大，A100用16
        lr0=0.005,
        lrf=0.01,
        patience=50,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.3,
        degrees=20.0,
        scale=0.6,
        flipud=0.5,
        fliplr=0.5,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        box=7.5, cls=0.5, dfl=1.5,
        project=f'{ROOT}/4_output',
        name='A_v8x_single',
        exist_ok=True
    )
    print("✅ v8x 训练完成")

# 验证
model = YOLO(V8X_WEIGHT)
metrics = model.val(data=f'{ROOT}/data_fixed.yaml')
print(f"\n📊 v8x 单模型性能:")
print(f"Val mAP50:    {metrics.box.map50:.4f}")
print(f"Val mAP50-95: {metrics.box.map:.4f}")

🚀 开始 YOLOv8x 极限训练...
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Wheat/data_fixed.yaml, degrees=20.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=A_v8x_single, nbs=64, nms=False, opset=None, optimi

In [8]:
# 【Cell 7】阶段4：5-Fold Ensemble —— 断联后可恢复到任意 Fold
import os, shutil, random
from pathlib import Path
from ultralytics import YOLO

ROOT = '/content/drive/MyDrive/Wheat'
OFFICIAL = f'{ROOT}/0_raw'

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

# ========== 准备 5-Fold 数据（幂等，已分过则跳过） ==========
FOLD_ROOT = f'{ROOT}/1_processed/folds'
os.makedirs(FOLD_ROOT, exist_ok=True)

a_train = sorted(Path(f'{OFFICIAL}/images/A_train').glob('*.png'))
random.seed(42)
random.shuffle(a_train)
fold_size = len(a_train) // 5
folds = [a_train[i*fold_size:(i+1)*fold_size] for i in range(5)]

# ========== 逐 Fold 训练（自动恢复） ==========
for fold in range(5):
    fold_out = f'{ROOT}/4_output/A_fold{fold}'
    fold_weight = f'{fold_out}/weights/best.pt'

    # 恢复检查：这个 Fold 是否已训完？
    if os.path.exists(fold_weight) and os.path.getsize(fold_weight) > 1000000:
        print(f"⚡️ Fold {fold}: 已存在，跳过")
        continue

    print(f"\n🚀 ========== Fold {fold}/4 ==========")

    # 准备该 Fold 的数据
    fold_dir = f'{FOLD_ROOT}/fold{fold}'
    for split in ['train', 'val']:
        os.makedirs(f'{fold_dir}/{split}/images', exist_ok=True)
        os.makedirs(f'{fold_dir}/{split}/labels', exist_ok=True)
        # 清理旧数据
        for f in Path(f'{fold_dir}/{split}/images').glob('*'):
            f.unlink()
        for f in Path(f'{fold_dir}/{split}/labels').glob('*'):
            f.unlink()

    val_imgs = folds[fold]
    train_imgs = [img for i, fl in enumerate(folds) if i != fold for img in fl]

    for img in train_imgs:
        shutil.copy(img, f'{fold_dir}/train/images/')
        lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{fold_dir}/train/labels/')

    for img in val_imgs:
        shutil.copy(img, f'{fold_dir}/val/images/')
        lbl = f'{OFFICIAL}/labels/A_train/{img.stem}.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{fold_dir}/val/labels/')

    # 写 yaml
    with open(f'{ROOT}/data_fold{fold}.yaml', 'w') as f:
        f.write(f"path: {fold_dir}\ntrain: train/images\nval: val/images\nnames:\n  0: wheat_head\nnc: 1")

    # 训练
    if os.path.exists(fold_out):
        shutil.rmtree(fold_out)

    model = YOLO('yolov8x.pt')
    model.train(
        data=f'{ROOT}/data_fold{fold}.yaml',
        epochs=120,
        imgsz=640,
        batch=16,
        lr0=0.005,
        patience=30,
        augment=True,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.3,
        degrees=20.0,
        scale=0.6,
        flipud=0.5,
        fliplr=0.5,
        project=f'{ROOT}/4_output',
        name=f'A_fold{fold}',
        exist_ok=True
    )

    print(f"✅ Fold {fold} 完成")

print("\n🏆 所有 Fold 训练完毕")


🚀 ========== Fold 0/4 ==========
Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Wheat/data_fold0.yaml, degrees=20.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=A_fold0, nbs=64, nms=False, opset=None,

In [9]:
# ═══════════════════════════════════════════════════════════
# 【修复版】过滤 width=0 或 height=0 的框
# ═══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import os, csv
from pathlib import Path
from PIL import Image
import torch
from torchvision.ops import nms
from ultralytics import YOLO

# ─── 路径 ───
ROOT = Path('/content/drive/MyDrive/Wheat')
A_TEST = ROOT / '0_raw/images/A_test'
MODEL_PATH = ROOT / '4_output/A_yolo11x_single/weights/best.pt'
csv_path = ROOT / '4_output/submission_A_single.csv'

assert A_TEST.exists()
assert MODEL_PATH.exists()

# ─── 加载模型 ───
print(f"加载模型: {MODEL_PATH}")
model = YOLO(str(MODEL_PATH))
models = [model]

# ─── 测试图片 ───
a_test = sorted(A_TEST.glob('*.png'))
print(f"\nA_test: {len(a_test)} 张")

# ─── 推理参数 ───
SCALES = [512, 640, 768]
CONF = 0.001
IOU = 0.5
MAX_BOXES_PER_SCALE = 300

all_results = []

for idx, img_path in enumerate(a_test):
    if idx % 100 == 0:
        print(f"  推理: {idx}/{len(a_test)}")

    image_id = img_path.stem
    with Image.open(img_path) as img:
        img_w, img_h = img.size

    boxes_list, scores_list = [], []

    for m in models:
        for scale in SCALES:
            r = m(
                str(img_path),
                imgsz=scale,
                conf=CONF,
                iou=IOU,
                augment=True,
                verbose=False
            )[0]

            if len(r.boxes) > 0:
                boxes = r.boxes.xyxy.cpu()
                scores = r.boxes.conf.cpu()
                if len(boxes) > MAX_BOXES_PER_SCALE:
                    topk = torch.topk(scores, MAX_BOXES_PER_SCALE)
                    boxes = boxes[topk.indices]
                    scores = scores[topk.indices]
                boxes_list.append(boxes)
                scores_list.append(scores)

    if len(boxes_list) > 0:
        all_boxes = torch.cat(boxes_list, dim=0)
        all_scores = torch.cat(scores_list, dim=0)
        keep = nms(all_boxes, all_scores, IOU)
        final_boxes = all_boxes[keep]
        final_scores = all_scores[keep]
    else:
        final_boxes = torch.empty((0, 4))
        final_scores = torch.empty(0)

    for i in range(len(final_boxes)):
        x1, y1, x2, y2 = final_boxes[i].tolist()

        # 绝对坐标 clip
        x1 = max(0.0, min(float(x1), float(img_w)))
        y1 = max(0.0, min(float(y1), float(img_h)))
        x2 = max(0.0, min(float(x2), float(img_w)))
        y2 = max(0.0, min(float(y2), float(img_h)))

        w = x2 - x1
        h = y2 - y1

        # ═══════════════════════════════════════════════════
        # 【核心修复】官网要求宽高必须严格大于 0
        # ═══════════════════════════════════════════════════
        if w > 0 and h > 0:
            x_center = (x1 + w / 2.0) / img_w
            y_center = (y1 + h / 2.0) / img_h
            width = w / img_w
            height = h / img_h
            confidence = float(final_scores[i])

            # 归一化后 clip
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            width = max(0.0, min(1.0, width))
            height = max(0.0, min(1.0, height))

            all_results.append({
                "image_id": image_id,
                "class_id": 0,
                "x_center": f"{x_center:.6f}",
                "y_center": f"{y_center:.6f}",
                "width": f"{width:.6f}",
                "height": f"{height:.6f}",
                "confidence": f"{confidence:.6f}"
            })

# ─── 保存 CSV（CRLF）───
csv_path.parent.mkdir(parents=True, exist_ok=True)
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['image_id', 'class_id', 'x_center', 'y_center',
                    'width', 'height', 'confidence'],
        lineterminator='\r\n'
    )
    writer.writeheader()
    writer.writerows(all_results)

print(f"\n🏆 CSV 已生成: {csv_path}")
print(f"   总行数: {len(all_results)}")

# ─── 快速验证：检查有无 width=0 或 height=0 ───
import pandas as pd
df = pd.read_csv(csv_path)
zero_wh = df[(df['width'] <= 0) | (df['height'] <= 0)]
print(f"   宽高为0的行数: {len(zero_wh)} (应为0)")
print("✅ 下载上传 Nexus")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
加载模型: /content/drive/MyDrive/Wheat/4_output/A_yolo11x_single/weights/best.pt

A_test: 1025 张
  推理: 0/1025
  推理: 100/1025
  推理: 200/1025
  推理: 300/1025
  推理: 400/1025
  推理: 500/1025
  推理: 600/1025
  推理: 700/1025
  推理: 800/1025
  推理: 900/1025
  推理: 1000/1025

🏆 CSV 已生成: /content/drive/MyDrive/Wheat/4_output/submission_A_single.csv
   总行数: 355246
   宽高为0的行数: 0 (应为0)
✅ 下载上传 Nexus
